<a href="https://colab.research.google.com/github/opyadav-07/machine-failure-prediction-using-machine-learning/blob/main/notebooks/02_binary_failure_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ***Importing Important Libraries***

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

 ***Loading processed data***

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Machine_Failure_Project/data/02_predictive_maintenance.csv')

In [ ]:
df.head()

,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF,Type_L,Type_M,Temperature_Difference,Power
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,False,True,10.5,66382.8
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,True,False,10.5,65190.4
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,True,False,10.4,74001.2
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,True,False,10.4,56603.5
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,True,False,10.5,56320.0


In [ ]:
df = df.astype(int)

In [ ]:
df.columns

Index(['Air temperature', 'Process temperature', 'Rotational speed', 'Torque',
       'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF',
       'Type_L', 'Type_M', 'Temperature_Difference', 'Power'],
      dtype='object')

# ***Train-Test-Split:***

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_df = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_df, columns=df.columns)


In [ ]:
leakage_columns = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

In [ ]:
X = df.drop(columns = ['Machine failure'] + leakage_columns, axis=1)
y = df['Machine failure'].astype(int)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

***Compare without and with class weights***


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

**Mode 1: Without class weights**

In [ ]:
LR_normal = LogisticRegression(
    max_iter=1000,
    random_state=42)
LR_normal.fit(X_train, y_train)
y_pred_normal = LR_normal.predict(X_test)

**Model 2: With class weights**

In [ ]:
LR_weighted = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42)
LR_weighted.fit(X_train, y_train)
y_pred_weighted = LR_weighted.predict(X_test)

In [ ]:
print("WITHOUT CLASS WEIGHTS")
print(classification_report(y_test, y_pred_normal))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_normal))

print("\n WITH CLASS WEIGHTS")
print(classification_report(y_test, y_pred_weighted))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_weighted))
print("\n Class weighting changes the trade-off between precision and recall, especially for the minority failure class")

WITHOUT CLASS WEIGHTS
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1932
           1       0.56      0.21      0.30        68

    accuracy                           0.97      2000
   macro avg       0.77      0.60      0.64      2000
weighted avg       0.96      0.97      0.96      2000

Confusion Matrix:
 [[1921   11]
 [  54   14]]

 WITH CLASS WEIGHTS
              precision    recall  f1-score   support

           0       0.99      0.85      0.92      1932
           1       0.17      0.85      0.28        68

    accuracy                           0.85      2000
   macro avg       0.58      0.85      0.60      2000
weighted avg       0.97      0.85      0.90      2000

Confusion Matrix:
 [[1644  288]
 [  10   58]]

 Class weighting changes the trade-off between precision and recall, especially for the minority failure class


# ***Handling Imbalance***

# **Approach A** — *Class weights*

In [ ]:
!pip install xgboost lightgbm


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


**Using:**
* class_weight='balanced' for Logistic Regression and Random Forest to account for class imbalance.
* For XGBoost, use scale_pos_weight
* For LightGBM, use class_weight

**Training the Four models using CLASS_WEIGHT:**

In [ ]:
# Calculate scale_pos_weight ratio (negative cases / positive cases)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000,
                                              class_weight='balanced',
                                              random_state=42),


    'Random Forest': RandomForestClassifier(n_estimators=300,
                                            class_weight='balanced',
                                            random_state=42,
                                            n_jobs=-1),


    # 'Support Vector Machine': SVC(class_weight='balanced',
    #                               random_state=42,
    #                       kernel='rbf'),  --> do not use in this models as it doesn't have (y_prob = model.predict_proba(X_test)[:, 1])



     'XGBoost': XGBClassifier(n_estimators=300,
                              max_depth=5,
                              learning_rate=0.05,
                              scale_pos_weight=ratio,
                              eval_metric="logloss",
                              random_state=42,
                              n_jobs=-1,),


    'LightGBM': LGBMClassifier( n_estimators=300,
                                learning_rate=0.05,
                                class_weight="balanced",
                                random_state=42,
                                verbosity=-1)

}

**Evaluate and compare models**

In [ ]:
from sklearn.metrics import( accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score, average_precision_score)

In [ ]:
results = []
trained_model = {}

for name, model in models.items():
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  y_prob = model.predict_proba(X_test)[:, 1]

  trained_model[name] = model

  results.append({
      'Model': name,
      'Accuracy': accuracy_score(y_test, y_pred),
      'Precision': precision_score(y_test, y_pred,zero_division=0),
      'Recall': recall_score(y_test, y_pred,zero_division=0),
      'F1 Score': f1_score(y_test, y_pred,zero_division=0),
      'ROC AUC': roc_auc_score(y_test, y_prob),
      'PR-AUC': average_precision_score(y_test, y_prob)
  })

# Convert results list to DataFrame
results_df = pd.DataFrame(results)

# Sort models by PR-AUC in descending order
results_df = results_df.sort_values(
    by='PR-AUC',
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

# Select the best model
best_model_name = results_df.loc[0, 'Model']
best_model = trained_model[best_model_name]

print("Best model:", best_model_name)
print("PR-AUC:", results_df.loc[0, 'PR-AUC'])
print("Recall:", results_df.loc[0, 'Recall'])
print("Precision:", results_df.loc[0, 'Precision'])
print("F1-score:", results_df.loc[0, 'F1 Score'])


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,PR-AUC
0,LightGBM,0.9800,0.7000,0.7206,0.7101,0.9714,0.8067
1,XGBoost,0.9755,0.6044,0.8088,0.6918,0.9761,0.7903
2,Random Forest,0.9845,0.8936,0.6176,0.7304,0.9702,0.7800
3,Logistic Regression,0.8510,0.1676,0.8529,0.2802,0.9319,0.4763


Best model: LightGBM
PR-AUC: 0.806681527173302
Recall: 0.7205882352941176
Precision: 0.7
F1-score: 0.7101449275362319


* LightGBM: Best PR-AUC, meaning it provides the strongest overall precision-recall ranking among these results.

* XGBoost: Strong alternative, with higher recall than LightGBM.

* Random Forest: Highest precision, but it misses more actual failures.

* Logistic Regression: High recall, but its low precision means many predicted failures are false alarms.

1. Choose **LightGBM** if your priority is the best **PR-AUC**(*Evaluates how well the model identifies rare failures*).

2. Consider **XGBoost** if detecting more failures is more important, since its recall is 0.8088 versus LightGBM's 0.7206.

In [ ]:
for name, model in trained_model.items():

    print(f"\n{'=' * 55}")
    print(name)
    print('=' * 55)

    y_pred = model.predict(X_test)

    print(classification_report(
        y_test,
        y_pred,
        target_names=["No Failure", "Failure"],
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Logistic Regression
              precision    recall  f1-score   support

  No Failure       0.99      0.85      0.92      1932
     Failure       0.17      0.85      0.28        68

    accuracy                           0.85      2000
   macro avg       0.58      0.85      0.60      2000
weighted avg       0.97      0.85      0.90      2000

Confusion Matrix:
[[1644  288]
 [  10   58]]

Random Forest
              precision    recall  f1-score   support

  No Failure       0.99      1.00      0.99      1932
     Failure       0.89      0.62      0.73        68

    accuracy                           0.98      2000
   macro avg       0.94      0.81      0.86      2000
weighted avg       0.98      0.98      0.98      2000

Confusion Matrix:
[[1927    5]
 [  26   42]]

XGBoost
              precision    recall  f1-score   support

  No Failure       0.99      0.98      0.99      1932
     Failure       0.60      0.81      0.69        68

    accuracy                           0.98    

**HYPERPARAMETER TUNING:**

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [ ]:
# Calculate imbalance ratio from training data only
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

GridSearchCV for XGBoost

In [ ]:
XGB = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=1)

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=XGB,
    param_grid=xgb_params,
    scoring='average_precision',
    cv=cv,
    verbose=1,
    n_jobs=-1,)


xgb_grid.fit(X, y)
result_xgb_grid = pd.DataFrame(xgb_grid.cv_results_)


print("Best XGBoost Parameters:")
print(xgb_grid.best_params_)

print("\nBest XGBoost CV PR-AUC:")
print(round(xgb_grid.best_score_, 4))


Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best XGBoost Parameters:
{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}

Best XGBoost CV PR-AUC:
0.8012


**Best Hyperparametrer for XGBoost:**

{'learning_rate': 0.1,
 'max_depth': 5,
 'n_estimators': 200,
 'subsample': 1.0}

With PR-AUC: 0.8012

In [ ]:
# result_xgb_grid

In [ ]:
lgbm = LGBMClassifier(
    objective='binary',
    class_weight='balanced',
    random_state=42,
    verbosity=-1,
    n_jobs=1
)

lgbm_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'num_leaves': [15, 31],
    'max_depth': [-1, 10],
    'min_child_samples': [20, 40]
}

lgbm_grid = GridSearchCV(
    estimator=lgbm,
    param_grid=lgbm_params,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)



lgbm_grid.fit(X, y)
result_lgbm_grid = pd.DataFrame(xgb_grid.cv_results_)


print("Best LightGBM Parameters:")
print(lgbm_grid.best_params_)

print("\nBest LightGBM CV PR-AUC:")
print(round(lgbm_grid.best_score_, 4))


Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best LightGBM Parameters:
{'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}

Best LightGBM CV PR-AUC:
0.8196


In [ ]:
#result_lgbm_grid

**Best Hyperparameter for LightGBM:**
{'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'n_estimators': 200,
 'num_leaves': 15}

 With PR-AUC: 0.8196



# **Approach B** — *SMOTE*

In [ ]:
!pip install imbalanced-learn

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Machine failure', axis=1)
y = df['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Check class distribution before SMOTE

In [ ]:
print("Before SMOTE:")
print(y_train.value_counts())

Before SMOTE:
Machine failure
0    7729
1     271
Name: count, dtype: int64


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42,
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [ ]:
print("After SMOTE:")
print(y_train_smote.value_counts())

After SMOTE:
Machine failure
0    7729
1    7729
Name: count, dtype: int64


Training Model on X_train_smote, y_train_smote

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model_Smote = RandomForestClassifier(    # not use --> class_weight='balanced'
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model_Smote.fit(X_train_smote, y_train_smote)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

In [ ]:
y_pred_smote = rf_model_Smote.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall   :", recall_score(y_test, y_pred_smote))
print("F1 Score :", f1_score(y_test, y_pred_smote))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))


Accuracy : 0.9805
Precision: 0.6435643564356436
Recall   : 0.9558823529411765
F1 Score : 0.7692307692307693

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1932
           1       0.64      0.96      0.77        68

    accuracy                           0.98      2000
   macro avg       0.82      0.97      0.88      2000
weighted avg       0.99      0.98      0.98      2000


Confusion Matrix:
[[1896   36]
 [   3   65]]


## Compare Normal vs Class Weight vs SMOTE

**Experiment 1: Baseline**

In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)
y_pred_baseline = rf_baseline.predict(X_test)


**Experiment 2: Class Weight**

In [ ]:
rf_weighted = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_weighted.fit(X_train, y_train)
y_pred_weighted = rf_weighted.predict(X_test)

Experiment 3: SMOTE

In [ ]:
rt_smote = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rt_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = rt_smote.predict(X_test)


**Comparision Table**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = pd.DataFrame({
    'Approach': [
        'Baseline',
        'Class Weights',
        'SMOTE'
    ],

    'Accuracy': [
        accuracy_score(y_test, y_pred_baseline),
        accuracy_score(y_test, y_pred_weighted),
        accuracy_score(y_test, y_pred_smote)
    ],

    'Precision': [
        precision_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_weighted),
        precision_score(y_test, y_pred_smote)
    ],

    'Recall': [
        recall_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_weighted),
        recall_score(y_test, y_pred_smote)
    ],

    'F1 Score': [
        f1_score(y_test, y_pred_baseline),
        f1_score(y_test, y_pred_weighted),
        f1_score(y_test, y_pred_smote)
    ]
})

results

,Approach,Accuracy,Precision,Recall,F1 Score
0,Baseline,0.9990,1.000000,0.970588,0.985075
1,Class Weights,0.9985,1.000000,0.955882,0.977444
2,SMOTE,0.9805,0.643564,0.955882,0.769231


**Save your best model**

In [ ]:
import os
import joblib

# Create the models folder inside Google Drive to avoid FileNotFoundError
os.makedirs("/content/drive/MyDrive/Machine_Failure_Project/models", exist_ok=True)

# Save best model
joblib.dump(
    rf_baseline,
    "/content/drive/MyDrive/Machine_Failure_Project/models/best_binary_model.pkl"
)

# Save feature names and order
joblib.dump(
    list(X_train.columns),
    "/content/drive/MyDrive/Machine_Failure_Project/models/feature_columns.pkl"
)

print("Best model and feature columns saved successfully!")

Best model and feature columns saved successfully!


**Save model comparison**

In [ ]:


comparison = pd.DataFrame({
    "Approach": ["Baseline", "Class Weights", "SMOTE"],
    "Accuracy": [0.9990, 0.9985, 0.9805],
    "Precision": [1.000000, 1.000000, 0.643564],
    "Recall": [0.970588, 0.955882, 0.955882],
    "F1 Score": [0.985075, 0.977444, 0.769231]
})

comparison.to_csv(
    "/content/drive/MyDrive/Machine_Failure_Project/models/model_comparison.csv",
    index=False
)